In [1]:
import torch
import torch.nn as nn

In [57]:
class MultiLatentAttention(nn.Module):
    def __init__(self, embed_dim, latent_dim, num_heads, max_seq_len = 512, dropout=0.0):
        super().__init__()
        self.embed_dim = embed_dim
        self.latent_dim = latent_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.WDKV = nn.Linear(embed_dim, latent_dim)
        self.WDK = nn.Linear(latent_dim, embed_dim)
        self.WDV = nn.Linear(latent_dim, embed_dim)
        self.WQ = nn.Linear(embed_dim, embed_dim)

        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer("mask", torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal = 1), persistent = False)
        self.register_buffer("cache_udkv", torch.empty(0), persistent = False)
        self.ptr_current_seq = 0

    def forward(self, X, use_cache = False):
        #CKV Cache implementation temporarily deferred
        batch_size, seq_len, embed_dim = X.size()
        initial_pos = self.ptr_current_seq

        Q = self.WQ(X) #Q = X * WQ
        UDKV = self.WDKV(X) #UDKV = X * WDKV

        if use_cache:
            self.cache_udkv = torch.cat([self.cache_udkv, UDKV], dim = 1)
            UDKV = self.cache_udkv
        self.ptr_current_seq += seq_len

        K = self.WDK(UDKV) #K = UDKV * WDK
        V = self.WDV(UDKV)  #V = UDKV * WDV

        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, self.ptr_current_seq, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, self.ptr_current_seq, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.embed_dim ** 0.5) # attn_scores = Q * K^T / sqrt(embed_dim)
        print(attn_scores.shape)
        print(self.mask[initial_pos : initial_pos + seq_len, :initial_pos + seq_len].shape)
        attn_scores = attn_scores.masked_fill(
                        self.mask[initial_pos : initial_pos + seq_len, :initial_pos + seq_len].bool(), torch.inf * (-1)
                        )
        attn_weights = nn.functional.softmax(attn_scores, dim = -1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous()
        context = context.view(batch_size, seq_len, embed_dim)
        context = self.out_proj(context)
        return context

In [58]:
X = torch.randn((2,4,8))

In [59]:
attn_layer = MultiLatentAttention(8, 4, 4)
attn_out = attn_layer(X, use_cache = False)

torch.Size([2, 4, 4, 4])
torch.Size([4, 4])


In [60]:
attn_layer = MultiLatentAttention(8, 8, 4)
attn_out = attn_layer(X, use_cache = True)

torch.Size([2, 4, 4, 4])
torch.Size([4, 4])


In [61]:
X1 = torch.randn((2,1,8))
attn_2_out = attn_layer(X1, use_cache = True)

torch.Size([2, 4, 1, 5])
torch.Size([1, 5])


In [47]:
attn_2_out.shape

torch.Size([2, 1, 8])

In [3]:
# Create a 4x4 matrix
A = torch.tensor([[ 1,  2,  3,  4],
                  [ 5,  6,  7,  8],
                  [ 9, 10, 11, 12],
                  [13, 14, 15, 16]])

# Default upper triangular (diagonal=0)
result = torch.triu(A)

In [4]:
A.masked_fill()

tensor([[ 1,  2,  3,  4],
        [ 0,  6,  7,  8],
        [ 0,  0, 11, 12],
        [ 0,  0,  0, 16]])

In [6]:
a = torch.ones(1,7)
torch.cat([a, torch.empty(0)])

tensor([[1., 1., 1., 1., 1., 1., 1.]])